# Day 03 — GroupBy Mastery
**Estimated time:** 60–75 min  
**Datasets:** `sales_orders.csv`, `materials_inventory.csv`

## Learning Objectives
- Use `.groupby()` with single columns, multiple columns, and custom aggregations
- Apply named aggregations for clean, self-documenting output
- Distinguish `transform` (row-level, preserves shape) from `aggregate` (reduces rows)


In [ ]:
import pandas as pd
import numpy as np

from analytics_bootcamp.config import RAW_DATA_DIR as DATA_DIR
sales = pd.read_csv(f"{DATA_DIR}/sales_orders.csv")
sales["ERDAT"] = pd.to_datetime(sales["ERDAT"], errors="coerce")
sales["NETWR"] = pd.to_numeric(sales["NETWR"], errors="coerce")
sales["MENGE"] = pd.to_numeric(sales["MENGE"], errors="coerce")

inv = pd.read_csv(f"{DATA_DIR}/materials_inventory.csv")
inv["LABST"] = pd.to_numeric(inv["LABST"], errors="coerce")
inv["STPRS"] = pd.to_numeric(inv["STPRS"], errors="coerce")
inv["LAST_MOVEMENT_DATE"] = pd.to_datetime(inv["LAST_MOVEMENT_DATE"], errors="coerce")

print("Sales:", sales.shape)
print("Inventory:", inv.shape)


In [ ]:
# ── 1. Single-column groupby ─────────────────────────────────────────────────
revenue_by_vkorg = (sales.groupby("VKORG")["NETWR"]
                         .sum()
                         .sort_values(ascending=False)
                         .rename("total_revenue")
                         .reset_index())
print(revenue_by_vkorg)


In [ ]:
# ── 2. Multi-column groupby ──────────────────────────────────────────────────
# Revenue by sales org + distribution channel (mirrors SAP InfoObject combos)
rev_by_channel = (sales.groupby(["VKORG","VTWEG"])["NETWR"]
                        .agg(["sum","count","mean"])
                        .rename(columns={"sum":"total_revenue","count":"order_count","mean":"avg_order_value"})
                        .sort_values("total_revenue", ascending=False)
                        .reset_index())
display(rev_by_channel.head(20))


In [ ]:
# ── 3. Named aggregations (pandas >= 0.25) ────────────────────────────────────
# Cleaner and self-documenting — preferred in production code
customer_summary = (
    sales.groupby("KUNNR")
    .agg(
        total_revenue=("NETWR", "sum"),
        order_count=("VBELN", "count"),
        avg_order_value=("NETWR", "mean"),
        min_date=("ERDAT", "min"),
        max_date=("ERDAT", "max"),
        open_orders=("STATUS", lambda x: (x == "OPEN").sum()),
        cancelled_orders=("STATUS", lambda x: (x == "CANCELLED").sum()),
    )
    .sort_values("total_revenue", ascending=False)
    .reset_index()
)
display(customer_summary.head(15))


In [ ]:
# ── 4. Multiple aggregation functions on multiple columns ────────────────────
multi_agg = (
    sales.groupby("SPART")
    .agg(
        rev_sum=("NETWR", "sum"),
        rev_mean=("NETWR", "mean"),
        rev_median=("NETWR", "median"),
        qty_sum=("MENGE", "sum"),
        n_orders=("VBELN", "count"),
        n_customers=("KUNNR", "nunique"),
    )
    .sort_values("rev_sum", ascending=False)
)
display(multi_agg)


In [ ]:
# ── 5. transform vs aggregate — KEY DISTINCTION ─────────────────────────────
# aggregate → REDUCES rows (one row per group)
# transform → PRESERVES shape (broadcasts group value back to each row)

# Use case: add a "revenue share" column to each order row
sales["vkorg_total_revenue"] = sales.groupby("VKORG")["NETWR"].transform("sum")
sales["order_revenue_share_pct"] = (sales["NETWR"] / sales["vkorg_total_revenue"] * 100).round(4)

print("Transform example — per-row revenue share within sales org:")
display(sales[["VBELN","KUNNR","VKORG","NETWR","vkorg_total_revenue","order_revenue_share_pct"]].head(10))


In [ ]:
# ── 6. Groupby with filter (groups satisfying a condition) ────────────────────
# Find sales orgs with > 5 customers and > $500k revenue
qualified_orgs = (
    sales.groupby("VKORG")
    .filter(lambda g: (g["KUNNR"].nunique() > 5) and (g["NETWR"].sum() > 500_000))
)
print(f"Records in qualifying orgs: {len(qualified_orgs):,}")
print("Qualifying VKORGs:", qualified_orgs["VKORG"].unique())


In [ ]:
# ── 7. Inventory groupby: stock value by material type and plant ──────────────
inv["INV_VALUE"] = inv["LABST"] * inv["STPRS"]

inv_summary = (
    inv.groupby(["MATERIAL_TYPE","WERKS"])
    .agg(
        n_materials=("MATNR","nunique"),
        total_stock_qty=("LABST","sum"),
        total_inv_value=("INV_VALUE","sum"),
        avg_stprs=("STPRS","mean"),
    )
    .sort_values("total_inv_value", ascending=False)
    .reset_index()
)
display(inv_summary.head(20))


---
## Daily Prompt

**Calculate total revenue, average order value, and cancellation rate by customer (`KUNNR`). Sort by total revenue descending. Then identify the top 10 customers by revenue and check if their cancellation rate is above or below the global average.**

```python
# YOUR SOLUTION
global_cancel_rate = (sales["STATUS"] == "CANCELLED").mean()
print(f"Global cancellation rate: {global_cancel_rate:.2%}")

customer_stats = (
    sales.groupby("KUNNR")
    .agg(
        total_revenue=("NETWR", "sum"),
        avg_order_value=("NETWR", "mean"),
        # YOUR SOLUTION: cancellation_rate
    )
    .sort_values("total_revenue", ascending=False)
)
# YOUR SOLUTION: flag customers above/below global_cancel_rate
```

> **Hint:** `cancellation_rate = ("STATUS", lambda x: (x == "CANCELLED").mean())` in `.agg()`.  
> Then add a column: `customer_stats["high_cancel_risk"] = customer_stats["cancellation_rate"] > global_cancel_rate`.
